# 【notebook｜N-13】GRPO 最小实现
> 对应：《06-RL-RLVR/01-RL与RLVR原理.md》【文档｜DOC-RL】；代码 `F-b06-grpo`

- 对应理论：`06-RL-RLVR/01-RL与RLVR原理.md` §6（GRPO 组内相对优势）
- 对应源码：`app/policy_server.py:416` /ttt_step；`nanoproof/nanoproof/rl.py:983`
- Colab：运行时选 CPU 即可（仅需 torch，几秒到几分钟）；有 GPU 自动用。

## 理论回顾

同一 prompt 采样一组 $G$ 个回答，奖励 $r_i$：

$$\hat A_i = \frac{r_i - \bar r}{\operatorname{std}(r) + \epsilon}, \qquad L = -\min\!\big(\rho \hat A,\ \operatorname{clip}(\rho,1\pm\epsilon)\hat A\big) + \beta D_{\mathrm{KL}}.$$

任务：单位数加法 `a+b=`，奖励 = 可验证的精确匹配。

In [1]:
# 【N-13.c1】
import torch, torch.nn as nn, torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0); gen = torch.Generator().manual_seed(0)
V, BLOCK = 14, 8
class TinyLM(nn.Module):
    def __init__(self, d=64, nhead=4, layers=2):
        super().__init__()
        self.tok = nn.Embedding(V, d); self.pos = nn.Embedding(BLOCK, d)
        layer = nn.TransformerEncoderLayer(d, nhead, 4*d, batch_first=True, dropout=0.0)
        self.enc = nn.TransformerEncoder(layer, layers); self.head = nn.Linear(d, V)
    def forward(self, idx):
        T = idx.size(1)
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))[None]
        m = torch.triu(torch.ones(T, T, device=idx.device), 1).bool()
        return self.head(self.enc(x, mask=m))
def last_logp(model, idx, tok):
    return F.log_softmax(model(idx)[:, -1, :], -1).gather(-1, tok).squeeze(-1)
def sample_token(model, prompt, temp=1.0):
    lp = F.log_softmax(model(prompt)[:, -1, :] / temp, -1)
    tok = torch.multinomial(lp.exp(), 1)
    return tok, lp.gather(-1, tok).squeeze(-1)
policy = TinyLM().to(device); ref = TinyLM().to(device)
ref.load_state_dict(policy.state_dict()); ref.eval()
for p in ref.parameters(): p.requires_grad_(False)
opt = torch.optim.AdamW(policy.parameters(), lr=3e-3)
print('device', device, '| params', sum(p.numel() for p in policy.parameters()))

device cuda | params 102286


In [2]:
# 【N-13.c2】
def make_batch(B):
    a = torch.randint(0,10,(B,),generator=gen); b = torch.randint(0,10,(B,),generator=gen)
    p = torch.stack([torch.tensor([int(x),10,int(y),11]) for x,y in zip(a,b)]).to(device)
    return p, (a+b) % 10
def grpo_step(G=8, B=8, clip=0.2, beta=0.01):
    prompt, ans = make_batch(B)
    rep = prompt.repeat_interleave(G, 0)
    tok, logp_old = sample_token(policy, rep)
    with torch.no_grad(): logp_ref = last_logp(ref, rep, tok)
    logp = last_logp(policy, rep, tok)
    corr = (tok.squeeze(-1).view(B,G) == ans.to(device)[:,None].expand(-1,G)).float()
    adv = ((corr - corr.mean(1,keepdim=True)) / (corr.std(1,keepdim=True)+1e-4)).view(-1)
    ratio = torch.exp(logp - logp_old)
    pg = -torch.min(ratio*adv, torch.clamp(ratio,1-clip,1+clip)*adv).mean()
    kl = beta*((logp-logp_ref)**2).mean()
    loss = pg + kl
    opt.zero_grad(); loss.backward()
    torch.nn.utils.clip_grad_norm_(policy.parameters(),1.0); opt.step()
    return float(loss), float(corr.mean())
for step in range(1, 201):
    loss, r = grpo_step()
    if step % 20 == 0 or step == 1: print(f'step {step:3d} | loss {loss:+.4f} | reward {r:.3f}')

/tmp/ipykernel_17020/605130624.py:19: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /app/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:838.)
  return float(loss), float(corr.mean())


step   1 | loss -0.0000 | reward 0.078
step  20 | loss +0.0005 | reward 0.031


step  40 | loss +0.0001 | reward 0.094
step  60 | loss +0.0000 | reward 0.094


step  80 | loss +0.0000 | reward 0.062
step 100 | loss +0.0000 | reward 0.109


step 120 | loss +0.0000 | reward 0.047
step 140 | loss +0.0000 | reward 0.031


step 160 | loss +0.0000 | reward 0.109
step 180 | loss +0.0000 | reward 0.062


step 200 | loss +0.0000 | reward 0.109


In [3]:
# 【N-13.c3】
@torch.no_grad()
def evaluate(n=200):
    p, ans = make_batch(n)
    return float((policy(p)[:, -1, :].argmax(-1) == ans.to(device)).float().mean())
print('greedy accuracy =', evaluate())
# 对照：随机基线约 10%。若偏低，增大步数或 G；若 nan，降低 lr。

greedy accuracy = 0.0


## 思考题
1. 一组奖励全相等时 `std=0`，优势与梯度会怎样？
2. 把 $G$ 改成 1 会怎样？为什么必须 $G>1$？
3. 把 $\beta$ 设为 0 会更快还是更不稳？
4. 与 `/ttt_step` 的 `-r(logp-logp_old)` 相比，clip 的区别在哪？

## 预期输出
`device cuda/cpu`；训练日志 `reward` 从 ~0.1 上升；`greedy accuracy` 通常 0.3–0.8。